In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import shutil
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import suite2p



In [ ]:
from suite2p.parameters import SETTINGS, default_settings

In [ ]:
settings_default = default_settings()




In [ ]:
def update_two_level_dict(d1: dict, d2: dict) -> dict:
    """
    Update d1 with values from d2, respecting two-level dict structure:
    - Non-dict fields in d1 are overwritten by the corresponding field in d2 (if present).
    - Dict fields in d1 are updated at the second level with the corresponding
      second-level dict in d2 (if present).
    Returns the updated d1.
    """
    for key, val in d2.items():
        if key in d1 and isinstance(d1[key], dict):
            if isinstance(val, dict):
                d1[key].update(val)
        else:
            d1[key] = val
    return d1

In [ ]:
def is_cuda_available() -> bool:
    """Returns True if a CUDA-capable GPU is available, False otherwise."""
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        pass

    try:
        import cupy
        cupy.cuda.runtime.getDeviceCount()
        return cupy.cuda.runtime.getDeviceCount() > 0
    except Exception:
        pass

    return False

In [ ]:
data_json_file = "/docs/data_suite2p.json"
with open(data_json_file, "r") as f:
    data = json.load(f)

In [ ]:
params_file = Path("/docs") / "params_default_suite2p.json"


with open(params_file, "r") as f:
    settings = json.load(f)

if "torch_device" not in settings:
    settings["torch_device"] = "cuda" if is_cuda_available() else "cpu"

settings = update_two_level_dict(settings_default, settings)

In [ ]:
root_path = Path(data["root_path"])
data_entries = data["data"]

# Check if entries are files or directories
first_entry = root_path / data_entries[0]

db = {}

if first_entry.is_file() or not first_entry.exists():
    # Entries are files — copy them into a new folder
    collected_folder = root_path / "collected_data"
    collected_folder.mkdir(exist_ok=True)

    for entry in data_entries:
        src = root_path / entry
        dst = collected_folder / src.name
        shutil.copy2(src, dst)
        print(f"Copied {src} -> {dst}")

    db["data_path"] = [str(collected_folder.resolve())]

else:
    # Entries are subfolders
    db["data_path"] = [str(root_path.resolve())]
    db["subfolders"] = data_entries

In [ ]:

print("db:", db)

In [ ]:
db["fast_disk"] = data["temp_dir"]
db["delete_bin"] = False
db["move_bin"] = True
db["save_folder"] = data["results_root_dir"]

In [ ]:
for key in ['fs', 'tau', 'nplanes', 'nchannels', 'functional_chan', 'force_sktiff', 'ignore_flyback', 'keep_movie_raw']:
    if key not in db:
        db[key] = settings[key]

In [ ]:
settings_out = suite2p.run_s2p(db, settings)


In [ ]:
os.listdir(Path(db['save_folder']) / 'combined')

In [ ]:
ts

In [ ]:
stats_combined = np.load(Path(db['save_folder']) / 'combined' / 'stat.npy', allow_pickle=True)

In [ ]:
del stats